# Notebook 6: ACRE Recommendation Algorithm

**Author:** Daniella Tahchi  
**Date:** 2024  
**Purpose:** Implement and demonstrate the Adaptive Cross-Cluster Recommendation Engine (ACRE) algorithm

**Input:** 
- Clustering artifacts from Notebook 05
- Feature matrix from Notebook 03
- User's selected favorite movies (3-5 movies)

**Output:** 
- Cross-cluster recommendations (10-15 movies)
- User preference centroid
- Cluster assignment and neighboring cluster analysis
- Recommendation metadata and similarity scores

---

## Objectives

1. **Load clustering artifacts** and feature engineering outputs
2. **Simulate user input** by selecting 3-5 favorite movies
3. **Build user preference centroid** using weighted averaging
4. **Assign user to optimal cluster** based on centroid distance
5. **Identify neighboring clusters** ranked by proximity
6. **Generate cross-cluster recommendations** from top N neighboring clusters
7. **Compute recommendation metrics** (similarity, diversity, novelty)
8. **Visualize results** with interactive cluster maps and movie displays

---

## Table of Contents

1. [Setup & Configuration](#1-setup--configuration)
2. [Load Artifacts](#2-load-artifacts)
3. [User Input Simulation](#3-user-input-simulation)
4. [Build User Preference Centroid](#4-build-user-preference-centroid)
5. [Assign User to Cluster](#5-assign-user-to-cluster)
6. [Identify Neighboring Clusters](#6-identify-neighboring-clusters)
7. [Generate ACRE Recommendations](#7-generate-acre-recommendations)
8. [Compute Recommendation Metrics](#8-compute-recommendation-metrics)
9. [Visualize Results](#9-visualize-results)
10. [Save ACRE Session](#10-save-acre-session)

---

# 1. Setup & Configuration

In [7]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Scikit-learn
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.decomposition import PCA

# Utilities
import warnings
from pathlib import Path
from datetime import datetime
import joblib
import json
from typing import List, Dict, Tuple

warnings.filterwarnings('ignore')

# Visualization settings
sns.set_style('whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("="*80)
print("ACRE RECOMMENDATION ALGORITHM - DEMONSTRATION")
print("="*80)
print(f"\nNotebook executed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n✓ Libraries loaded successfully\n")

ACRE RECOMMENDATION ALGORITHM - DEMONSTRATION

Notebook executed: 2026-04-17 15:42:03

✓ Libraries loaded successfully



## Configuration Parameters

In [8]:
# Configuration
CONFIG = {
    # Input paths
    'clustering_dir': Path('../artifacts/clustering'),
    'features_dir': Path('../artifacts/feature-engineering'),
    'preprocessing_dir': Path('../artifacts/preprocessing'),
    
    # Output paths
    'output_dir': Path('../artifacts/acre-recommendations'),
    
    # ACRE parameters
    'n_neighboring_clusters': 5,  # Number of neighboring clusters to retrieve from
    'n_movies_per_cluster': 3,    # Number of recommendations per neighboring cluster
    'use_weighted_centroid': True, # Weight by vote_average when building user centroid
    
    # Display parameters
    'poster_base_url': 'https://image.tmdb.org/t/p/w500',
    'random_state': 42
}

# Create output directory
CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()}, indent=2))
print("\n✓ Configuration loaded")

Configuration:
{
  "clustering_dir": "..\\artifacts\\clustering",
  "features_dir": "..\\artifacts\\feature-engineering",
  "preprocessing_dir": "..\\artifacts\\preprocessing",
  "output_dir": "..\\artifacts\\acre-recommendations",
  "n_neighboring_clusters": 5,
  "n_movies_per_cluster": 3,
  "use_weighted_centroid": true,
  "poster_base_url": "https://image.tmdb.org/t/p/w500",
  "random_state": 42
}

✓ Configuration loaded


---

# 2. Load Artifacts

In [9]:
print("="*80)
print("LOADING CLUSTERING AND FEATURE ARTIFACTS")
print("="*80)

# ──────────────────────────────────────────
# Load clustering artifacts
# ──────────────────────────────────────────
print("\n[1] Loading clustering model...")
clustering_model = joblib.load(CONFIG['clustering_dir'] / 'clustering_model.joblib')
print(f"✓ Loaded: {type(clustering_model).__name__}")

print("\n[2] Loading cluster centroids...")
cluster_centroids = np.load(CONFIG['clustering_dir'] / 'cluster_centroids.npy')
n_clusters = len(cluster_centroids)
print(f"✓ Loaded {n_clusters} cluster centroids (shape: {cluster_centroids.shape})")

print("\n[3] Loading cluster assignments...")
cluster_assignments = pd.read_csv(CONFIG['clustering_dir'] / 'cluster_assignments.csv')
print(f"✓ Loaded cluster assignments for {len(cluster_assignments):,} movies")

print("\n[4] Loading dimensionality reduction model...")
reduction_model = joblib.load(CONFIG['clustering_dir'] / 'pca_model.joblib')

# ──────────────────────────────────────────
# Load feature engineering artifacts
# ──────────────────────────────────────────
print("\n[5] Loading feature matrix...")
feature_matrix = np.load(CONFIG['features_dir'] / 'feature_matrix.npy')
print(f"✓ Feature matrix loaded: {feature_matrix.shape}")

print("\n[6] Loading movie index...")
movie_index = pd.read_csv(CONFIG['features_dir'] / 'movie_index.csv')
print(f"✓ Movie index loaded: {len(movie_index):,} movies")

# ──────────────────────────────────────────
# Load cleaned movie data (for metadata)
# ──────────────────────────────────────────
print("\n[7] Loading cleaned movie data...")
movies_df = pd.read_csv(CONFIG['preprocessing_dir'] / 'cleaned_movies.csv')
print(f"✓ Loaded {len(movies_df):,} movies with metadata")

# ──────────────────────────────────────────
# Apply dimensionality reduction if used
# ──────────────────────────────────────────
if reduction_model is not None:
    print(f"\n[8] Applying PCA transformation to feature matrix...")
    feature_matrix_reduced = reduction_model.transform(feature_matrix)
    print(f"✓ Reduced feature matrix: {feature_matrix_reduced.shape}")
else:
    feature_matrix_reduced = feature_matrix
    print(f"\n[8] Using full feature matrix (no reduction)")

print("\n" + "="*80)
print("✅ ALL ARTIFACTS LOADED SUCCESSFULLY")
print("="*80)

LOADING CLUSTERING AND FEATURE ARTIFACTS

[1] Loading clustering model...
✓ Loaded: KMeans

[2] Loading cluster centroids...
✓ Loaded 40 cluster centroids (shape: (40, 30))

[3] Loading cluster assignments...
✓ Loaded cluster assignments for 27,777 movies

[4] Loading dimensionality reduction model...

[5] Loading feature matrix...
✓ Feature matrix loaded: (27777, 172)

[6] Loading movie index...
✓ Movie index loaded: 27,777 movies

[7] Loading cleaned movie data...
✓ Loaded 27,777 movies with metadata

[8] Applying PCA transformation to feature matrix...
✓ Reduced feature matrix: (27777, 30)

✅ ALL ARTIFACTS LOADED SUCCESSFULLY


---

# 3. User Input Simulation

## Display Random Sample of Movies for Selection

In a production web app, the user would search and select movies via the UI.  
For this demonstration, we'll display a random sample and manually specify movie IDs.

In [10]:
# Display random sample of popular movies for demonstration
popular_movies = movies_df[movies_df['vote_count'] >= 1000].sample(50, random_state=CONFIG['random_state'])

print("\n" + "="*80)
print("SAMPLE MOVIES FOR SELECTION")
print("="*80 + "\n")

display_cols = ['id', 'title', 'genres', 'release_year', 'vote_average', 'vote_count']
print(popular_movies[display_cols].to_string(index=False))

print("\n" + "="*80)


SAMPLE MOVIES FOR SELECTION

    id                                   title                                       genres  release_year  vote_average  vote_count
 12101                           Soylent Green                    Science Fiction, Thriller          1973         6.900        1134
181533 Night at the Museum: Secret of the Tomb           Adventure, Comedy, Fantasy, Family          2014         6.200        5495
   769                              GoodFellas                                 Drama, Crime          1990         8.464       11726
   754                                Face/Off               Action, Crime, Science Fiction          1997         7.019        4826
 68817                               Footloose                        Drama, Music, Romance          2011         6.597        1557
273248                       The Hateful Eight                      Drama, Mystery, Western          2015         7.746       13291
270487                           Hail, Caesar!

## Specify User's Favorite Movies

**Instructions:** Replace the movie IDs below with IDs from the sample above (or any valid movie IDs from the dataset).

In [11]:
# USER INPUT: Specify 3-5 favorite movie IDs
# Example: User who likes sci-fi and thrillers
user_selected_movie_ids = [
    27205,  # Inception (2010)
    157336, # Interstellar (2014)
    155,    # The Dark Knight (2008)
    260513  # Arrival (2016)
]

# Validate selection
assert len(user_selected_movie_ids) >= 3, "Please select at least 3 movies"
assert len(user_selected_movie_ids) <= 5, "Please select at most 5 movies"

# Get selected movies details
selected_movies = movies_df[movies_df['id'].isin(user_selected_movie_ids)]

print("\n" + "="*80)
print("USER'S SELECTED FAVORITE MOVIES")
print("="*80 + "\n")

print(selected_movies[['id', 'title', 'genres', 'release_year', 'vote_average']].to_string(index=False))

print("\n" + "="*80)
print(f"✓ {len(user_selected_movie_ids)} movies selected")
print("="*80)


USER'S SELECTED FAVORITE MOVIES

    id           title                               genres  release_year  vote_average
 27205       Inception   Action, Science Fiction, Adventure          2010         8.364
157336    Interstellar    Adventure, Drama, Science Fiction          2014         8.417
   155 The Dark Knight       Drama, Action, Crime, Thriller          2008         8.512
260513   Incredibles 2 Action, Adventure, Animation, Family          2018         7.481

✓ 4 movies selected


---

# 4. Build User Preference Centroid

## Weighted Centroid Computation

The user's preference centroid is computed as the **weighted average** of their selected movies' feature vectors, where the weights are the **normalized vote_average** scores.

This ensures that higher-rated movies in the user's selection have more influence on the centroid.

In [12]:
print("\n" + "="*80)
print("BUILDING USER PREFERENCE CENTROID")
print("="*80)

# Get indices of selected movies in the feature matrix
selected_indices = movie_index[movie_index['id'].isin(user_selected_movie_ids)].index.tolist()

# Extract feature vectors (use reduced matrix if dimensionality reduction was applied)
selected_vectors = feature_matrix_reduced[selected_indices]

print(f"\n✓ Extracted {len(selected_vectors)} feature vectors")
print(f"✓ Vector dimension: {selected_vectors.shape[1]}")

if CONFIG['use_weighted_centroid']:
    print("\n[Weighted Averaging]")
    
    # Get vote averages for selected movies
    selected_ratings = selected_movies.set_index('id').loc[user_selected_movie_ids, 'vote_average'].values
    
    # Normalize weights
    weights = selected_ratings / selected_ratings.sum()
    
    print("\nMovie weights:")
    for movie_id, rating, weight in zip(user_selected_movie_ids, selected_ratings, weights):
        title = selected_movies[selected_movies['id'] == movie_id]['title'].values[0]
        print(f"  • {title[:40]:40s} | Rating: {rating:.1f} | Weight: {weight:.3f}")
    
    # Compute weighted centroid
    user_centroid = np.average(selected_vectors, axis=0, weights=weights)
    
else:
    print("\n[Simple Averaging]")
    user_centroid = np.mean(selected_vectors, axis=0)

print(f"\n✓ User preference centroid computed (shape: {user_centroid.shape})")
print("="*80)


BUILDING USER PREFERENCE CENTROID

✓ Extracted 4 feature vectors
✓ Vector dimension: 30

[Weighted Averaging]

Movie weights:
  • Inception                                | Rating: 8.4 | Weight: 0.255
  • Interstellar                             | Rating: 8.4 | Weight: 0.257
  • The Dark Knight                          | Rating: 8.5 | Weight: 0.260
  • Incredibles 2                            | Rating: 7.5 | Weight: 0.228

✓ User preference centroid computed (shape: (30,))


---

# 5. Assign User to Cluster

In [13]:
print("\n" + "="*80)
print("ASSIGNING USER TO CLUSTER")
print("="*80)

# Compute Euclidean distances from user centroid to all cluster centroids
distances_to_centroids = euclidean_distances([user_centroid], cluster_centroids)[0]

# Assign user to nearest cluster
user_cluster = np.argmin(distances_to_centroids)

print(f"\n✓ User assigned to Cluster {user_cluster}")
print(f"✓ Distance to assigned cluster centroid: {distances_to_centroids[user_cluster]:.4f}")

# Get cluster statistics
cluster_size = len(cluster_assignments[cluster_assignments['cluster'] == user_cluster])
print(f"✓ Cluster {user_cluster} contains {cluster_size:,} movies")

# Sample movies from user's cluster
cluster_movies = cluster_assignments[cluster_assignments['cluster'] == user_cluster].merge(
    movies_df[['id', 'title', 'genres', 'vote_average']], 
    on='id'
).sample(min(10, cluster_size), random_state=CONFIG['random_state'])

print(f"\nSample movies from Cluster {user_cluster}:")
print(cluster_movies[['title_x', 'genres_x', 'vote_average_x']].to_string(index=False))

print("\n" + "="*80)


ASSIGNING USER TO CLUSTER

✓ User assigned to Cluster 26
✓ Distance to assigned cluster centroid: 0.9660
✓ Cluster 26 contains 361 movies

Sample movies from Cluster 26:
                       title_x                               genres_x  vote_average_x
                        Xtreme               Action, Adventure, Crime           6.432
Crouching Tiger, Hidden Dragon      Adventure, Drama, Action, Romance           7.409
         The Magnificent Seven             Adventure, Action, Western           6.418
               War of the Dead              Horror, Action, Adventure           4.205
       Brotherhood of the Wolf     Adventure, Horror, Action, History           6.712
               The Comancheros    Romance, Western, Action, Adventure           6.744
             Where Eagles Dare                 Action, Adventure, War           7.549
                  Red Cliff II Action, Adventure, Drama, History, War           7.314
              The Naked Jungle      Action, Adventure, 

---

# 6. Identify Neighboring Clusters

In [14]:
print("\n" + "="*80)
print("IDENTIFYING NEIGHBORING CLUSTERS")
print("="*80)

# Sort clusters by distance (excluding user's own cluster)
cluster_distance_ranking = np.argsort(distances_to_centroids)
cluster_distance_ranking = cluster_distance_ranking[cluster_distance_ranking != user_cluster]

# Determine number of neighboring clusters to use
N = min(CONFIG['n_neighboring_clusters'], len(cluster_distance_ranking))

# Select top N neighboring clusters
neighboring_clusters = cluster_distance_ranking[:N]

print(f"\n✓ Identified {N} neighboring clusters")
print(f"\nNeighboring cluster ranking (by distance to user centroid):\n")

neighboring_cluster_info = []
for rank, cluster_id in enumerate(neighboring_clusters, 1):
    distance = distances_to_centroids[cluster_id]
    size = len(cluster_assignments[cluster_assignments['cluster'] == cluster_id])
    
    neighboring_cluster_info.append({
        'rank': rank,
        'cluster_id': cluster_id,
        'distance': distance,
        'size': size
    })
    
    print(f"  {rank}. Cluster {cluster_id:2d} | Distance: {distance:.4f} | Size: {size:,} movies")

neighboring_cluster_df = pd.DataFrame(neighboring_cluster_info)

print("\n" + "="*80)


IDENTIFYING NEIGHBORING CLUSTERS

✓ Identified 5 neighboring clusters

Neighboring cluster ranking (by distance to user centroid):

  1. Cluster  3 | Distance: 0.9885 | Size: 431 movies
  2. Cluster 30 | Distance: 1.1351 | Size: 529 movies
  3. Cluster 22 | Distance: 1.1966 | Size: 309 movies
  4. Cluster 27 | Distance: 1.2786 | Size: 796 movies
  5. Cluster 16 | Distance: 1.3965 | Size: 284 movies



---

# 7. Generate ACRE Recommendations

## Cross-Cluster Retrieval

For each neighboring cluster:
1. Retrieve all movies in the cluster
2. Exclude user's selected movies (if any are in the cluster)
3. Compute cosine similarity to user centroid
4. Select top N most similar movies

In [15]:
print("\n" + "="*80)
print("GENERATING ACRE RECOMMENDATIONS")
print("="*80)

acre_recommendations = []

for cluster_id in neighboring_clusters:
    print(f"\n[Cluster {cluster_id}]")
    
    # Get all movies in this cluster
    cluster_movie_ids = cluster_assignments[cluster_assignments['cluster'] == cluster_id]['id'].values
    
    # Remove user's selected movies if any are in this cluster
    cluster_movie_ids = np.setdiff1d(cluster_movie_ids, user_selected_movie_ids)
    
    # Get indices in feature matrix
    cluster_indices = movie_index[movie_index['id'].isin(cluster_movie_ids)].index.tolist()
    
    # Extract feature vectors
    cluster_vectors = feature_matrix_reduced[cluster_indices]
    
    # Compute cosine similarity to user centroid
    similarities = cosine_similarity([user_centroid], cluster_vectors)[0]
    
    # Get top N most similar
    top_n_indices_in_cluster = np.argsort(similarities)[::-1][:CONFIG['n_movies_per_cluster']]
    top_n_similarities = similarities[top_n_indices_in_cluster]
    
    # Map back to movie IDs
    top_n_movie_ids = cluster_movie_ids[top_n_indices_in_cluster]
    
    # Get movie details
    for movie_id, similarity in zip(top_n_movie_ids, top_n_similarities):
        movie_details = movies_df[movies_df['id'] == movie_id].iloc[0]
        
        acre_recommendations.append({
            'movie_id': movie_id,
            'title': movie_details['title'],
            'genres': movie_details['genres'],
            'release_year': movie_details['release_year'],
            'vote_average': movie_details['vote_average'],
            'vote_count': movie_details['vote_count'],
            'poster_path': movie_details.get('poster_path', ''),
            'source_cluster': cluster_id,
            'similarity_to_user': similarity
        })
    
    print(f"  ✓ Retrieved {len(top_n_movie_ids)} recommendations (avg similarity: {top_n_similarities.mean():.3f})")

acre_recommendations_df = pd.DataFrame(acre_recommendations)

print(f"\n{'='*80}")
print(f"✅ ACRE RECOMMENDATIONS GENERATED")
print(f"{'='*80}")
print(f"\n✓ Total recommendations: {len(acre_recommendations_df)}")
print(f"✓ From {N} neighboring clusters ({CONFIG['n_movies_per_cluster']} movies per cluster)")
print(f"\n{'='*80}")


GENERATING ACRE RECOMMENDATIONS

[Cluster 3]
  ✓ Retrieved 3 recommendations (avg similarity: 0.813)

[Cluster 30]
  ✓ Retrieved 3 recommendations (avg similarity: 0.629)

[Cluster 22]
  ✓ Retrieved 3 recommendations (avg similarity: 0.593)

[Cluster 27]
  ✓ Retrieved 3 recommendations (avg similarity: 0.626)

[Cluster 16]
  ✓ Retrieved 3 recommendations (avg similarity: 0.621)

✅ ACRE RECOMMENDATIONS GENERATED

✓ Total recommendations: 15
✓ From 5 neighboring clusters (3 movies per cluster)



## Display ACRE Recommendations

In [16]:
print("\n" + "="*80)
print("ACRE RECOMMENDATIONS (Cross-Cluster)")
print("="*80 + "\n")

display_cols = ['title', 'genres', 'release_year', 'vote_average', 'source_cluster', 'similarity_to_user']
print(acre_recommendations_df[display_cols].to_string(index=False))

print("\n" + "="*80)


ACRE RECOMMENDATIONS (Cross-Cluster)

                                 title                                       genres  release_year  vote_average  source_cluster  similarity_to_user
                        Wing Commander           Science Fiction, Action, Adventure          1999         4.774               3            0.823946
Star Trek VI: The Undiscovered Country Science Fiction, Action, Adventure, Thriller          1991         7.000               3            0.815731
                              Returner           Action, Adventure, Science Fiction          2002         6.524               3            0.799829
                              THX 1138               Science Fiction, Action, Drama          1971         6.453              30            0.655350
     Battle for the Planet of the Apes                      Action, Science Fiction          1973         5.704              30            0.623546
                            Blown Away                                   

---

# 8. Compute Recommendation Metrics

In [17]:
print("\n" + "="*80)
print("COMPUTING RECOMMENDATION METRICS")
print("="*80)

# ──────────────────────────────────────────
# 1. Mean Cosine Similarity
# ──────────────────────────────────────────
mean_similarity = acre_recommendations_df['similarity_to_user'].mean()
print(f"\n[1] Mean Cosine Similarity: {mean_similarity:.4f}")
print(f"    (Higher = closer to user preferences)")

# ──────────────────────────────────────────
# 2. Intra-List Diversity (ILD)
# ──────────────────────────────────────────
# Average pairwise cosine distance among recommended movies
rec_indices = movie_index[movie_index['id'].isin(acre_recommendations_df['movie_id'])].index.tolist()
rec_vectors = feature_matrix_reduced[rec_indices]

pairwise_similarities = cosine_similarity(rec_vectors)
# Get upper triangle (exclude diagonal)
upper_triangle_indices = np.triu_indices_from(pairwise_similarities, k=1)
pairwise_distances = 1 - pairwise_similarities[upper_triangle_indices]

intra_list_diversity = pairwise_distances.mean()
print(f"\n[2] Intra-List Diversity (ILD): {intra_list_diversity:.4f}")
print(f"    (Higher = more diverse recommendations)")

# ──────────────────────────────────────────
# 3. Novelty
# ──────────────────────────────────────────
# Average Euclidean distance from user centroid
novelty_distances = euclidean_distances([user_centroid], rec_vectors)[0]
mean_novelty = novelty_distances.mean()

print(f"\n[3] Mean Novelty (Euclidean Distance): {mean_novelty:.4f}")
print(f"    (Higher = more novel/exploratory recommendations)")

# ──────────────────────────────────────────
# 4. Cluster Spread
# ──────────────────────────────────────────
unique_clusters = acre_recommendations_df['source_cluster'].nunique()
print(f"\n[4] Cluster Spread: {unique_clusters} clusters")
print(f"    (Recommendations sourced from {unique_clusters} different taste groups)")

# ──────────────────────────────────────────
# 5. Average Rating
# ──────────────────────────────────────────
avg_rating = acre_recommendations_df['vote_average'].mean()
print(f"\n[5] Average Vote Average: {avg_rating:.2f}")
print(f"    (Quality indicator of recommended movies)")

print("\n" + "="*80)

# Store metrics
acre_metrics = {
    'mean_similarity': float(mean_similarity),
    'intra_list_diversity': float(intra_list_diversity),
    'mean_novelty': float(mean_novelty),
    'cluster_spread': int(unique_clusters),
    'avg_rating': float(avg_rating),
    'n_recommendations': len(acre_recommendations_df)
}


COMPUTING RECOMMENDATION METRICS

[1] Mean Cosine Similarity: 0.6564
    (Higher = closer to user preferences)

[2] Intra-List Diversity (ILD): 0.6820
    (Higher = more diverse recommendations)

[3] Mean Novelty (Euclidean Distance): 1.5138
    (Higher = more novel/exploratory recommendations)

[4] Cluster Spread: 5 clusters
    (Recommendations sourced from 5 different taste groups)

[5] Average Vote Average: 6.43
    (Quality indicator of recommended movies)



---

# 9. Visualize Results

## Cluster Proximity Visualization

In [18]:
# Visualize cluster distances
cluster_dist_df = pd.DataFrame({
    'Cluster ID': range(n_clusters),
    'Distance to User': distances_to_centroids
})

cluster_dist_df['Type'] = 'Other'
cluster_dist_df.loc[cluster_dist_df['Cluster ID'] == user_cluster, 'Type'] = 'User Cluster'
cluster_dist_df.loc[cluster_dist_df['Cluster ID'].isin(neighboring_clusters), 'Type'] = 'Neighboring (Used)'

fig = px.bar(
    cluster_dist_df.sort_values('Distance to User'),
    x='Cluster ID',
    y='Distance to User',
    color='Type',
    title='Distance from User Centroid to All Cluster Centroids',
    labels={'Distance to User': 'Euclidean Distance'},
    color_discrete_map={
        'User Cluster': '#00CC96',
        'Neighboring (Used)': '#FFA15A',
        'Other': '#E0E0E0'
    }
)

fig.update_layout(height=500, showlegend=True)
fig.show()

## Recommendation Similarity Distribution

In [19]:
fig = px.histogram(
    acre_recommendations_df,
    x='similarity_to_user',
    nbins=20,
    color='source_cluster',
    title='Distribution of Recommendation Similarity Scores by Source Cluster',
    labels={'similarity_to_user': 'Cosine Similarity to User Centroid', 'count': 'Number of Movies'},
    opacity=0.7
)

fig.add_vline(x=mean_similarity, line_dash="dash", line_color="red",
              annotation_text=f"Mean: {mean_similarity:.3f}", annotation_position="top right")

fig.update_layout(height=400)
fig.show()

## 2D Cluster Visualization with User Position

In [20]:
# Apply PCA to cluster centroids for 2D visualization
pca_2d = PCA(n_components=2, random_state=CONFIG['random_state'])
centroids_2d = pca_2d.fit_transform(cluster_centroids)
user_centroid_2d = pca_2d.transform([user_centroid])[0]

# Create DataFrame for plotting
centroids_plot_df = pd.DataFrame({
    'PC1': centroids_2d[:, 0],
    'PC2': centroids_2d[:, 1],
    'Cluster': range(n_clusters),
    'Type': ['User Cluster' if i == user_cluster else 'Neighboring' if i in neighboring_clusters else 'Other' 
             for i in range(n_clusters)]
})

# Plot
fig = px.scatter(
    centroids_plot_df,
    x='PC1',
    y='PC2',
    color='Type',
    text='Cluster',
    title='2D Cluster Map: User Position and Neighboring Clusters',
    labels={'PC1': 'Principal Component 1', 'PC2': 'Principal Component 2'},
    color_discrete_map={
        'User Cluster': '#00CC96',
        'Neighboring': '#FFA15A',
        'Other': '#E0E0E0'
    },
    size_max=15
)

# Add user centroid
fig.add_scatter(
    x=[user_centroid_2d[0]],
    y=[user_centroid_2d[1]],
    mode='markers',
    marker=dict(size=20, color='red', symbol='star', line=dict(width=2, color='black')),
    name='User Centroid',
    showlegend=True
)

fig.update_traces(textposition='top center')
fig.update_layout(height=600, width=800)
fig.show()

---

# 10. Save ACRE Session

In [21]:
print("\n" + "="*80)
print("SAVING ACRE SESSION")
print("="*80)


# ──────────────────────────────────────────
# 1. Save recommendations
# ──────────────────────────────────────────
recommendations_path = CONFIG['output_dir'] / 'acre_recommendations.csv'
acre_recommendations_df.to_csv(recommendations_path, index=False)
print(f"\n✓ Saved recommendations: {recommendations_path.name}")

# ──────────────────────────────────────────
# 2. Save session metadata
# ──────────────────────────────────────────
session_metadata = {
    'user_selected_movies': user_selected_movie_ids,
    'user_cluster': int(user_cluster),
    'neighboring_clusters': neighboring_clusters.tolist(),
    'n_recommendations': len(acre_recommendations_df),
    'metrics': acre_metrics,
    'config': {
        'n_neighboring_clusters': CONFIG['n_neighboring_clusters'],
        'n_movies_per_cluster': CONFIG['n_movies_per_cluster'],
        'use_weighted_centroid': CONFIG['use_weighted_centroid']
    }
}

metadata_path = CONFIG['output_dir'] / 'session_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(session_metadata, f, indent=4)
print(f"✓ Saved session metadata: {metadata_path.name}")

# ──────────────────────────────────────────
# 3. Save user centroid
# ──────────────────────────────────────────
centroid_path = CONFIG['output_dir'] / 'user_centroid.npy'
np.save(centroid_path, user_centroid)
print(f"✓ Saved user centroid: {centroid_path.name}")

print(f"\n{'='*80}")
print(f"✅ SESSION SAVED TO: {CONFIG['output_dir']}")
print(f"{'='*80}")


SAVING ACRE SESSION

✓ Saved recommendations: acre_recommendations.csv
✓ Saved session metadata: session_metadata.json
✓ Saved user centroid: user_centroid.npy

✅ SESSION SAVED TO: ..\artifacts\acre-recommendations


---

# Summary

## ACRE Algorithm Demonstration Complete

**What We Did:**

1. ✅ Loaded clustering artifacts and feature matrix
2. ✅ Simulated user input (selected 3-5 favorite movies)
3. ✅ Built weighted user preference centroid
4. ✅ Assigned user to optimal cluster based on centroid proximity
5. ✅ Identified top N neighboring clusters
6. ✅ Generated cross-cluster recommendations (10-15 movies)
7. ✅ Computed recommendation quality metrics
8. ✅ Visualized cluster relationships and recommendation distributions
9. ✅ Saved session for future analysis

**Key Results:**

- **User Cluster:** {user_cluster}
- **Recommendations Generated:** {len(acre_recommendations_df)}
- **Source Clusters:** {unique_clusters}
- **Mean Similarity:** {mean_similarity:.3f}
- **Intra-List Diversity:** {intra_list_diversity:.3f}
- **Mean Novelty:** {mean_novelty:.3f}

**Next Step:** Notebook 07 - Baseline Comparison (compare ACRE with same-cluster recommendations)